# Interactive Tutorial for the HWO High-Resolution Imager (HRI) 

[SYOTools](https://github.com/spacetelescope/hwo-tools) is a framework to enable users to create [Science Yield Optimization web tools](http://hwo.stsci.edu) for observatory design. It uses Bokeh & astropy to visualize exposure time calculators (ETCs), as well as other science cases, as a function of various observatory parameters. While SYOTools can be used independently of a particular observatory or science case, it has been primarily written to facilitate the design of [HWO](https://www.habitableworldsobservatory.org).

SYOTools is divided into two main parts, contained in the `syotools.models` and `syotools.interface` subpackages. This Jupyter Notebook is intended as a walkthrough for using `syotools.models` to perform calculations. A future tutorial on using `syotools.interface` to design a web tool is in preparation; however, it will not be a similar Jupyter Notebook, as the interface framework is highly integrated with Bokeh Server. Instead, we will use Bokeh's `output_notebook` function, along with IPython interactors (as described [here](https://github.com/bokeh/bokeh/blob/0.12.13/examples/howto/notebook_comms/Jupyter%20Interactors.ipynb)), to visualize the example calculations below. 


In [ ]:
#Import Bokeh interface tools
from ipywidgets import interact
from bokeh.io import push_notebook, show, output_notebook
from bokeh.plotting import figure
from bokeh.layouts import column
output_notebook()

#Import numpy, syotools, and astropy.units
### NOTE: you will likely receive some synphot warnings when executing these imports, these are expected
import numpy as np
from syotools.models import Telescope, Camera, Spectrograph, Source #models for the observatory and instruments
from syotools.utils.jsonunit import str_jsunit #for printing JsonUnit and JsonSpectrum wrappers in a readable way
from syotools.utils import pre_encode, pre_decode
from syotools.spectra.spec_defaults import syn_spectra_library
import astropy.units as u
import synphot as syn
import stsynphot as stsyn

#make sure that synphot can find its reference files!
import os
assert os.path.exists(os.environ["PYSYN_CDBS"])

## Example 1: HRI ETC

*"The High Resolution Imager (HRI) instrument is the primary astronomical imaging instrument for observations in the near UV through the near IR. The hri design provides a 2 x 3 arcminute field-of-view, taking full advantage of the angular resolution provided by the telescope, and consists of two channels - an ultraviolet-visible (UVIS) channel covering 200 nm - 950 nm and a near-infrared (NIR) channel covering the range 800 nm - 2200 nm. The respective focal plane detector arrays provide Nyquist sampled images at 400 nm (2.73 mas/pixel) for UVIS imaging and at 1200 nm (8.20 mas/pixel) for NIR imaging."* (adapted from LUVOIR STDT study) 

In this example, we will create a exposure time calculator for HRI, so that we can calculate the signal-to-noise ratio (SNR) for several possible template spectra. This approximates some of the functionality of [the official HRI ETC tool](http://hwo.stsci.edu/camera_etc).

In [ ]:
#Instatiate the observatory using syotools.models.Telescope
#This loads the default Telescope values, which are already based on the HWO design. 
#It also now loads all of the instruments (Cameras, Spectrographs, and IFSes) into a variable.
hwo_ex1 = Telescope()
hwo_ex1.set_from_hwome('EAC5')

# see all of the instruments
print(hwo_ex1.instruments.keys())

In [ ]:
#Instantiate the instrument, using syotools.models.Camera, and link it with the telescope
#This loads the default Camera values, which are already based on the HWO-HRI design
hri = hwo_ex1.instruments["HRI_S.HRI_S_UVIS_Imager"]
hri2 = hwo_ex1.instruments["HRI_S.HRI_S_NIR_Imager"]

In [ ]:
#Create a new photometric exposure for the camera
hri_exposure = hri.create_exposure()
hri2_exposure = hri2.create_exposure()

# set the kind of calculation you want done
hri_exposure.unknown = "snr"
hri2_exposure.unknown = "snr"


#Print the available wave bands
bandpass = hri.recover('bands')

pivotwave = hri.pivotwave.value
pivotwave2 = hri2.pivotwave.value

print("HRI wave bands:") 
print(hri.band)
for band in hri.bands:
    print("   {:3s} - {:5.2f} ± {:5.2f}".format(band, hri.configuration["band"][band]["bandpass"].pivot(), hri.configuration["band"][band]["bandwidth"])) 
for band in hri2.bands:
    print("   {:3s} - {:5.2f} ± {:5.2f}".format(band, hri2.configuration["band"][band]["bandpass"].pivot(), hri2.configuration["band"][band]["bandwidth"])) 


print(pivotwave, pivotwave2)

#Print the default template
default_hri_template = hri_exposure.source.name
print("Current SED template: {}".format(default_hri_template)) 

hri_template_codes = ['fab', 'o5v', 'b5v', 'g2v', 'm2v', 'orion', 'elliptical', 'sbc', 'starburst', 'ngc1068']
available_hri_templates = syn_spectra_library.keys()

In [ ]:
#Create the Bokeh SED figure
hri_sed = hri_exposure.recover('source').sed
hri_wave = hri_sed.waveset.to_value(u.nm)
hri_flux = syn.units.convert_flux(hri_wave, hri_sed(hri_wave), u.ABmag)
hri_sed_fig = figure(height=300, width=600, title="SED", x_axis_label="Wavelength [nm]",
                 y_axis_label="AB Mag", y_range=(35, 21), x_range=(800, 24000))
hri_sed_line = hri_sed_fig.line(hri_wave, hri_flux, color='orange', line_width=3)

In [ ]:
#Create the Bokeh SNR figure
#hri.band = None
hri_snr = hri_exposure.recover("snr")
hri2_snr = hri2_exposure.recover("snr")

hri_snr = [hri_snr[x].value for x in range(len(hri_snr))]    
hri2_snr = [hri2_snr[x].value for x in range(len(hri2_snr))]    

hri_snr_fig = figure(height=300, width=600, title="SNR", x_axis_label="Wavelength [nm]", 
                 y_axis_label="SNR", y_range=(0, 20), x_range=(800, 24000))
uv_hri_snr = hri_snr_fig.scatter(pivotwave, hri_snr, color='orange', line_width=3)
nir_hri_snr = hri_snr_fig.scatter(pivotwave2, hri2_snr, color='blue', line_width=3)

In [ ]:
#Define the update callback function for interactive inputs
def hri_update(template=default_hri_template, aperture=8., exptime=1., v_magnitude=30.):
    #find the correct template code
    sed_id = template
    
    #turn off calculations until everything is updated
    hri_exposure.disable()

    exptime = [exptime] * len(hri.bandnames)
    
    #update all of the telescope & exposure parameters
    hri_exposure.exptime = exptime * u.hr
    newsource = Source()
    newsource.set_sed(template, v_magnitude, 0, 0, bandpass="johnson,v")
    hri_exposure.source = newsource
    hwo_ex1.effective_diameter = aperture * u.m
    #turn calculations back on and recalculate based on updated parameters
    hri_exposure.enable()
    
    #recover the recalculated values, and make sure everything is in the right units
    hri_sed = hri_exposure.recover('source')
    hri_snr = hri_exposure.recover("snr")
    hri2_snr = hri2_exposure.recover("snr")
    
    hri_snr = [hri_snr[x].value for x in range(len(hri_snr))]
    hri2_snr = [hri2_snr[x].value for x in range(len(hri2_snr))]
    hri_sed = hri_sed.sed
    hri_wave = hri_sed.waveset.to_value(u.nm)
    hri_flux = syn.units.convert_flux(hri_wave, hri_sed(hri_wave), u.ABmag)
    
    #sanitize the sed fluxes because some of the synphot spectra don't play nice
    hri_flux[~np.isfinite(hri_flux)] = v_magnitude * u.ABmag
    
    #update the SED figure
    hri_sed_fig.y_range.start = hri_flux.max().value + 5
    hri_sed_fig.y_range.end = hri_flux.min().value - 5
    hri_sed_line.data_source.data = {'x': hri_wave, 'y': hri_flux}
    
    #update the SNR figure
    hri_snr_fig.y_range.start = 0.
    hri_snr_fig.y_range.end = max(1.3 * np.max(hri_snr), 5.)
    uv_hri_snr.data_source.data['y'] = hri_snr
    nir_hri_snr.data_source.data['y'] = hri2_snr
    
    #update the plots
    push_notebook()

In [ ]:
# Show the plots
# depending on your jupyter config you may need to shift+enter this cell 
# after changing the sliders below to see an updated result plot
hri_handle = show(column(hri_sed_fig, hri_snr_fig), notebook_handle=True)

In [ ]:
#Create the interactive inputs
#shift+enter here will reset to defaults. 
hri_inputs = interact(hri_update, template=available_hri_templates, aperture=(2.0, 12.0), exptime=(0.1, 10.0, 0.1), 
         v_magnitude=(20.0, 35.0, 0.1))